# Osa 2: Puheentunnistus Whisper-mallilla

## Kirjastot

In [ ]:
import re
import time
import torch
import librosa
import librosa.display
import numpy as np
import pandas as pd
import jiwer
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# pip install transformers librosa ipywidgets sentencepiece sacremoses jiwer gradio

## Ääninäytteet

In [ ]:
# viite = oikea litterointi (sanasta sanaan se, mitä äänitteessä sanotaan)
# kieli = "fi" tai "en"
NAYTTEET = [
    {"nimi": "selkea_fi", "tiedosto": "voices/selkea_fi.wav", "kieli": "fi",
     "kuvaus": "Selkeä yleiskielinen puhe, hiljainen huone, puhelimen mikrofoni, puhuja 1",
     "viite": "Suomen luonto on kauneimmillaan kesällä, kun yöt ovat valoisia ja järvet tyyniä. Moni suomalainen viettää loman mökillä uiden, saunoen ja kalastaen. Syksyllä metsät täyttyvät sienestäjistä ja marjastajista, ja ensimmäiset pakkaset tuovat mukanaan kirkkaat tähtitaivaat."},
    {"nimi": "haly_fi", "tiedosto": "voices/haly_fi.wav", "kieli": "fi",
     "kuvaus": "Sääennuste luettuna taustahälyn kanssa, puhuja 1",
     "viite": "Huomenna sää on enimmäkseen pilvinen, ja iltapäivällä sataa paikoin vettä. Lämpötila nousee noin kahdeksaantoista asteeseen. Viikonloppuna aurinko näyttäytyy, mutta yöllä lämpötila voi laskea lähelle nollaa. Muistakaa pukeutua lämpimästi, jos lähdette ulos."},
    {"nimi": "puhuja2_fi", "tiedosto": "voices/puhuja2_fi.wav", "kieli": "fi",
     "kuvaus": "Selkeä yleiskielinen puhe, sama teksti kuin selkea_fi, puhuja 2",
     "viite": "Suomen luonto on kauneimmillaan kesällä, kun yöt ovat valoisia ja järvet tyyniä. Moni suomalainen viettää loman mökillä uiden, saunoen ja kalastaen. Syksyllä metsät täyttyvät sienestäjistä ja marjastajista, ja ensimmäiset pakkaset tuovat mukanaan kirkkaat tähtitaivaat."},
    {"nimi": "murre_fi", "tiedosto": "voices/murre_fi.wav", "kieli": "fi",
     "kuvaus": "Puhekielinen suomi, hiljainen huone, puhuja 1",
     "viite": "Mä oon tänää menny kauppaan ja ostin maitoo, leipää ja vähän juustoo. Sit mä kävelin kotiin ja kattelin telkkarii koko illan. Huomenna ei oo koulua, joten voin nukkuu pitkään."},
    {"nimi": "selkea_en", "tiedosto": "voices/selkea_en.wav", "kieli": "en",
     "kuvaus": "Englanninkielinen puhe, hiljainen huone, puhuja 1",
     "viite": "Finland is known for its thousands of lakes, its forests and its long, dark winters. In the summer the sun barely sets in the north, and people gather around campfires to enjoy the light. Many visitors come to see the northern lights and to try a traditional sauna."},
]

# Whisper-mallit (Hugging Face), vaihtoehtoja: tiny, base, small, medium, large-v3
MALLIT = ["tiny", "base", "small"]

In [ ]:
# Load audio files (16 kHz, mono)
aanet = {}
for n in NAYTTEET:
    y, sr = librosa.load(n["tiedosto"], sr=16000)
    aanet[n["nimi"]] = y

    print(f"{n['nimi']}  ({n['tiedosto']})")
    display(Audio(y, rate=sr))
    print(f"Duration: {librosa.get_duration(y=y, sr=sr):.2f} seconds")
    print(f"Sample rate: {sr} Hz\n")

## Mel-spektrogrammi

In [ ]:
def piirra_aani(y, sr=16000, otsikko=""):
    # Plot waveform
    plt.figure(figsize=(14, 4))
    librosa.display.waveshow(y, sr=sr)
    plt.title(f'Audio Waveform - {otsikko}')
    plt.tight_layout()
    plt.show()

    # Create mel spectrogram
    plt.figure(figsize=(14, 5))
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, y_axis='mel', x_axis='time', sr=sr)
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Mel-frequency Spectrogram - {otsikko}')
    plt.tight_layout()
    plt.show()

for nimi, y in aanet.items():
    piirra_aani(y, otsikko=nimi)

## Mallien lataus

In [ ]:
# Valitse laite: NVIDIA-GPU, Apple-GPU tai CPU
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()
          else "cpu")
print(f"Using device: {device}")

mallit = {}
for nimi in MALLIT:
    model_name = f"openai/whisper-{nimi}"
    t0 = time.perf_counter()
    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
    mallit[nimi] = (processor, model)
    parametrit = sum(p.numel() for p in model.parameters())
    print(f"{model_name}: {parametrit/1e6:.0f} M parameters, load time {time.perf_counter() - t0:.1f} s")

## Tunnistus

In [ ]:
def tunnista(y, malli_nimi, kieli="fi", sr=16000):
    """Transcribe an audio array (< 30 s) with the chosen Whisper model. Returns (text, seconds)."""
    processor, model = mallit[malli_nimi]
    t0 = time.perf_counter()
    inputs = processor(y, sampling_rate=sr, return_tensors="pt").to(device, model.dtype)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, language=kieli, task="transcribe")
    teksti = processor.batch_decode(generated_ids, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=False)[0].strip()
    return teksti, time.perf_counter() - t0

rivit = []
for n in NAYTTEET:
    for m in MALLIT:
        teksti, aika = tunnista(aanet[n["nimi"]], m, n["kieli"])
        rivit.append({"nayte": n["nimi"], "malli": m, "aika_s": aika, "tunnistettu": teksti})
        print(f"[{m:>5}] {n['nimi']}: {teksti}")

tulokset = pd.DataFrame(rivit)

## Vertailu: tarkkuus

In [ ]:
def normalisoi(s):
    s = s.lower()
    s = re.sub(r"[^\w\s]", "", s)
    return re.sub(r"\s+", " ", s).strip()

viitteet = {n["nimi"]: n["viite"] for n in NAYTTEET}

def virhe(mitta, rivi):
    viite = viitteet[rivi["nayte"]]
    if not viite:
        return np.nan
    return mitta(normalisoi(viite), normalisoi(rivi["tunnistettu"]))

tulokset["WER"] = tulokset.apply(lambda r: virhe(jiwer.wer, r), axis=1)
tulokset["CER"] = tulokset.apply(lambda r: virhe(jiwer.cer, r), axis=1)

display(tulokset.pivot(index="nayte", columns="malli", values="WER")[MALLIT])
display(tulokset.pivot(index="nayte", columns="malli", values="CER")[MALLIT])

In [ ]:
tulokset.pivot(index="nayte", columns="malli", values="WER")[MALLIT].plot.bar(figsize=(10, 4))
plt.ylabel("WER")
plt.title("Word error rate by model")
plt.tight_layout()
plt.show()

## Vertailu: nopeus

In [ ]:
tulokset["kesto_s"] = tulokset["nayte"].map({k: len(y) / 16000 for k, y in aanet.items()})
tulokset["RTF"] = tulokset["aika_s"] / tulokset["kesto_s"]  # alle 1 = nopeampi kuin reaaliaika

display(tulokset.pivot(index="nayte", columns="malli", values="aika_s")[MALLIT])
display(tulokset.groupby("malli")[["aika_s", "RTF"]].mean().loc[MALLIT])

tulokset.pivot(index="nayte", columns="malli", values="aika_s")[MALLIT].plot.bar(figsize=(10, 4))
plt.ylabel("Time (s)")
plt.title("Transcription time by model")
plt.tight_layout()
plt.show()

## Tunnistetut tekstit rinnakkain

In [ ]:
for n in NAYTTEET:
    print("=" * 80)
    print(n["nimi"], "-", n["kuvaus"])
    print("REFERENCE:", n["viite"])
    for m in MALLIT:
        r = tulokset[(tulokset.nayte == n["nimi"]) & (tulokset.malli == m)].iloc[0]
        print(f"{m:>6}: {r['tunnistettu']}")

## Kohinakoe

In [ ]:
def lisaa_kohinaa(y, snr_db):
    teho_signaali = np.mean(y ** 2)
    teho_kohina = teho_signaali / (10 ** (snr_db / 10))
    return (y + np.random.normal(0, np.sqrt(teho_kohina), y.shape)).astype(np.float32)

KOHINA_NAYTE = NAYTTEET[0]
SNR_ARVOT = [20, 10, 5, 0, -5]

np.random.seed(0)
kohina_rivit = []
for snr in SNR_ARVOT:
    meluisa = lisaa_kohinaa(aanet[KOHINA_NAYTE["nimi"]], snr)
    for m in MALLIT:
        teksti, aika = tunnista(meluisa, m, KOHINA_NAYTE["kieli"])
        wer = jiwer.wer(normalisoi(KOHINA_NAYTE["viite"]), normalisoi(teksti)) if KOHINA_NAYTE["viite"] else np.nan
        kohina_rivit.append({"SNR_dB": snr, "malli": m, "WER": wer, "aika_s": aika, "tunnistettu": teksti})

kohina_tulokset = pd.DataFrame(kohina_rivit)
kohina_tulokset

In [ ]:
kohina_tulokset.pivot(index="SNR_dB", columns="malli", values="WER")[MALLIT].sort_index(ascending=False).plot(marker="o", figsize=(8, 4))
plt.xlabel("SNR (dB)")
plt.ylabel("WER")
plt.title("Effect of noise on recognition accuracy")
plt.gca().invert_xaxis()
plt.show()

piirra_aani(lisaa_kohinaa(aanet[KOHINA_NAYTE["nimi"]], 0), otsikko="SNR 0 dB")

## Kielen tunnistus ja englanninkielinen puhe

In [ ]:
def tunnista_kieli(y, malli_nimi, sr=16000):
    # Kieltä ei anneta mallille, jolloin Whisper valitsee kielen itse
    processor, model = mallit[malli_nimi]
    inputs = processor(y, sampling_rate=sr, return_tensors="pt").to(device, model.dtype)
    with torch.no_grad():
        ids = model.generate(**inputs, task="transcribe")
    teksti = processor.batch_decode(ids, skip_special_tokens=False)[0]
    osuma = re.search(r"<\|([a-z]{2,3})\|>", teksti)
    return osuma.group(1) if osuma else "?"

rivit = []
for n in NAYTTEET:
    for m in MALLIT:
        rivit.append({"nayte": n["nimi"], "oikea_kieli": n["kieli"], "malli": m,
                      "tunnistettu_kieli": tunnista_kieli(aanet[n["nimi"]], m)})
pd.DataFrame(rivit).pivot(index=["nayte", "oikea_kieli"], columns="malli", values="tunnistettu_kieli")[MALLIT]

In [ ]:
display(tulokset[tulokset.nayte.str.endswith("_en")][["nayte", "malli", "aika_s", "WER", "CER", "tunnistettu"]])
display(tulokset[tulokset.nayte.str.endswith("_fi")].groupby("malli")[["WER", "CER"]].mean().loc[MALLIT])

## Aikaleimat

In [ ]:
processor, model = mallit[MALLIT[-1]]
y = aanet[NAYTTEET[0]["nimi"]]
inputs = processor(y, sampling_rate=16000, return_tensors="pt").to(device, model.dtype)
with torch.no_grad():
    ids = model.generate(**inputs, language="fi", task="transcribe", return_timestamps=True)

for o in processor.tokenizer.decode(ids[0], output_offsets=True)["offsets"]:
    print(f"[{o['timestamp'][0]:6.2f} - {o['timestamp'][1]:6.2f}]  {o['text'].strip()}")

## Yhteenveto

In [ ]:
tulokset.groupby("malli")[["WER", "CER", "aika_s", "RTF"]].mean().loc[MALLIT]

## Gradio-käyttöliittymä

In [ ]:
import gradio as gr

GRADIO_MALLI = MALLIT[-1]

def transcribe_audio(audio_path):
    if audio_path is None:
        return "No audio recorded"
    y, sr = librosa.load(audio_path, sr=16000)
    teksti, _ = tunnista(y, GRADIO_MALLI, "fi")
    return teksti

demo = gr.Interface(
    fn=transcribe_audio,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
    outputs=gr.Textbox(lines=10),
    title="Speech to Text with Whisper",
    description="Record your voice or upload an audio file (< 30 s) to get it transcribed in Finnish"
)

In [ ]:
demo.launch()